## 1. Setup and Data Loading

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path().absolute().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from sklearn.model_selection import train_test_split

# Import project modules
from mediwatch.data.preprocess import PreprocessingArtifacts
from mediwatch.models.train import train_model, train_all_baselines, get_baseline_models
from mediwatch.models.evaluate import evaluate_model, compare_models, print_evaluation_report
from mediwatch.models.tune import run_hpo, tune_multiple_models
from mediwatch.registry.mlflow_registry import (
    setup_mlflow, get_best_run, compare_runs, register_model, promote_model_to_production
)

import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")

### Load Preprocessed Data

In [ ]:
# Load train and test data
data_dir = Path('../data/processed')

train_df = pd.read_csv(data_dir / 'train.csv')
test_df = pd.read_csv(data_dir / 'test.csv')

print(f"Train set: {train_df.shape}")
print(f"Test set: {test_df.shape}")

# Load preprocessing artifacts
artifacts = PreprocessingArtifacts.load(data_dir / 'preprocessing_artifacts.pkl')
print(f"\nPreprocessing artifacts loaded")
print(f"Target column: {artifacts.metadata['target_col']}")
print(f"Number of features: {artifacts.metadata['n_features']}")

### Prepare Features and Target

In [ ]:
# Get target column name
target_col = artifacts.metadata['target_col']

# Separate features and target
X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print(f"Training features: {X_train.shape}")
print(f"Training target: {y_train.shape}")
print(f"\nTest features: {X_test.shape}")
print(f"Test target: {y_test.shape}")

# Check class distribution
print(f"\nTarget distribution (train):")
print(y_train.value_counts())
print(f"\nTarget distribution (%):")
print(y_train.value_counts(normalize=True) * 100)

## 2. MLflow Setup

In [ ]:
# Setup MLflow
# Note: Start MLflow server first with: python scripts/setup_mlflow.py --action start
# Or run in background: mlflow server --host 127.0.0.1 --port 5000

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
EXPERIMENT_NAME = "mediwatch-readmission"

try:
    experiment_id = setup_mlflow(
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=EXPERIMENT_NAME
    )
    print(f"✓ MLflow configured")
    print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")
    print(f"  Experiment: {EXPERIMENT_NAME}")
    print(f"  Experiment ID: {experiment_id}")
    print(f"\nAccess MLflow UI at: {MLFLOW_TRACKING_URI}")
except Exception as e:
    print(f"Warning: Could not connect to MLflow server: {e}")
    print(f"\nTo start MLflow server, run in a separate terminal:")
    print(f"  python scripts/setup_mlflow.py --action start")
    print(f"\nContinuing without MLflow tracking...")

## 3. Baseline Models Training

In [ ]:
# Check available models
available_models = get_baseline_models()
print(f"Available baseline models: {list(available_models.keys())}")

### Train Individual Models

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_artifacts = train_model(
    x_train=X_train,
    y_train=y_train,
    model_name="logistic_regression",
    use_scaling=True,
    track_mlflow=True,
    experiment_name=EXPERIMENT_NAME
)

# Evaluate on test set
lr_metrics = evaluate_model(
    pipeline=lr_artifacts.pipeline,
    x_test=X_test,
    y_test=y_test,
    track_mlflow=True
)

print_evaluation_report(lr_metrics, "Logistic Regression")

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_artifacts = train_model(
    x_train=X_train,
    y_train=y_train,
    model_name="random_forest",
    use_scaling=False,  # Tree-based models don't need scaling
    track_mlflow=True,
    experiment_name=EXPERIMENT_NAME
)

# Evaluate
rf_metrics = evaluate_model(
    pipeline=rf_artifacts.pipeline,
    x_test=X_test,
    y_test=y_test,
    track_mlflow=True
)

print_evaluation_report(rf_metrics, "Random Forest")

In [ ]:
# Train XGBoost (if available)
try:
    print("Training XGBoost...")
    xgb_artifacts = train_model(
        x_train=X_train,
        y_train=y_train,
        model_name="xgboost",
        use_scaling=False,
        track_mlflow=True,
        experiment_name=EXPERIMENT_NAME
    )
    
    xgb_metrics = evaluate_model(
        pipeline=xgb_artifacts.pipeline,
        x_test=X_test,
        y_test=y_test,
        track_mlflow=True
    )
    
    print_evaluation_report(xgb_metrics, "XGBoost")
    
except Exception as e:
    print(f"XGBoost not available: {e}")

### Train All Baseline Models

In [ ]:
# Train all available baseline models
baseline_results = train_all_baselines(
    x_train=X_train,
    y_train=y_train,
    use_scaling=True,  # Will be applied selectively based on model type
    track_mlflow=True,
    experiment_name=EXPERIMENT_NAME
)

## 4. Model Comparison

In [ ]:
# Evaluate all models on test set
pipelines = {name: artifacts.pipeline for name, artifacts in baseline_results.items()}

comparison_df = compare_models(
    models=pipelines,
    x_test=X_test,
    y_test=y_test
)

print("\nModel Comparison:")
display(comparison_df)

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# F1 Score comparison
axes[0].barh(comparison_df['model'], comparison_df['f1'])
axes[0].set_xlabel('F1 Score')
axes[0].set_title('Model Comparison - F1 Score')
axes[0].set_xlim(0, 1)

# ROC AUC comparison
if 'roc_auc' in comparison_df.columns:
    axes[1].barh(comparison_df['model'], comparison_df['roc_auc'])
    axes[1].set_xlabel('ROC AUC')
    axes[1].set_title('Model Comparison - ROC AUC')
    axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

## 5. Hyperparameter Optimization with Ray Tune

### Tune Single Model

In [ ]:
# Tune Random Forest (example)
print("Starting hyperparameter optimization for Random Forest...")
print("This may take several minutes...\n")

hpo_results = run_hpo(
    x_train=X_train,
    y_train=y_train,
    model_name="random_forest",
    num_samples=20,  # Number of trials
    max_concurrent_trials=4,
    cv=3,  # 3-fold cross-validation
    use_scaling=False,
    mlflow_experiment_name=EXPERIMENT_NAME
)

print(f"\nBest configuration:")
for key, value in hpo_results['best_config'].items():
    print(f"  {key}: {value}")
print(f"\nBest CV F1 score: {hpo_results['best_score']:.4f}")

In [ ]:
# Train final model with best hyperparameters
best_rf_artifacts = train_model(
    x_train=X_train,
    y_train=y_train,
    model_name="random_forest",
    model_params=hpo_results['best_config'],
    use_scaling=False,
    track_mlflow=True,
    experiment_name=EXPERIMENT_NAME
)

# Evaluate tuned model
best_rf_metrics = evaluate_model(
    pipeline=best_rf_artifacts.pipeline,
    x_test=X_test,
    y_test=y_test,
    track_mlflow=True
)

print_evaluation_report(best_rf_metrics, "Random Forest (Tuned)")

### Tune Multiple Models (Optional)

In [ ]:
# Tune multiple models
# Warning: This can take a long time!

# Uncomment to run:
# multi_hpo_results = tune_multiple_models(
#     x_train=X_train,
#     y_train=y_train,
#     model_names=["random_forest", "gradient_boosting", "xgboost"],
#     num_samples=10,  # Fewer trials per model
#     max_concurrent_trials=2,
#     cv=3,
#     mlflow_experiment_name=EXPERIMENT_NAME
# )

print("Multiple model tuning skipped (uncomment to run)")

## 6. MLflow Experiment Analysis

In [ ]:
# Compare all runs in MLflow
runs_df = compare_runs(
    experiment_name=EXPERIMENT_NAME,
    metrics=["test_f1", "test_accuracy", "test_precision", "test_recall", "test_roc_auc"],
    tracking_uri=MLFLOW_TRACKING_URI
)

print("\nAll MLflow Runs:")
display(runs_df.head(10))

In [ ]:
# Get best run
best_run = get_best_run(
    experiment_name=EXPERIMENT_NAME,
    metric="test_f1",
    tracking_uri=MLFLOW_TRACKING_URI
)

if best_run:
    print(f"\nBest Run Details:")
    print(f"  Run ID: {best_run.info.run_id}")
    print(f"  Run Name: {best_run.data.tags.get('mlflow.runName', 'N/A')}")
    print(f"\nMetrics:")
    for metric, value in best_run.data.metrics.items():
        print(f"  {metric}: {value:.4f}")

## 7. Model Registration and Deployment Preparation

In [ ]:
# Register best model
if best_run:
    model_uri = f"runs:/{best_run.info.run_id}/model"
    
    model_version = register_model(
        model_uri=model_uri,
        model_name="mediwatch_readmission_classifier",
        tags={
            "task": "readmission_prediction",
            "framework": "sklearn",
            "best_metric": "f1_score",
        },
        description="Best performing model for patient readmission prediction"
    )
    
    print(f"\nModel registered: mediwatch_readmission_classifier v{model_version.version}")

In [ ]:
# Promote to production
promote_model_to_production(
    model_name="mediwatch_readmission_classifier",
    tracking_uri=MLFLOW_TRACKING_URI
)

## 8. Summary and Next Steps

In [ ]:
print("=" * 80)
print("MODEL TRAINING SUMMARY")
print("=" * 80)

print(f"\nDataset:")
print(f"  Training samples: {X_train.shape[0]:,}")
print(f"  Test samples: {X_test.shape[0]:,}")
print(f"  Features: {X_train.shape[1]}")

print(f"\nModels Trained:")
for name in baseline_results.keys():
    print(f"  ✓ {name}")

print(f"\nBest Model (by F1 score):")
best_model_name = comparison_df.iloc[0]['model']
best_f1 = comparison_df.iloc[0]['f1']
print(f"  Model: {best_model_name}")
print(f"  F1 Score: {best_f1:.4f}")

print(f"\nMLflow:")
print(f"  Experiment: {EXPERIMENT_NAME}")
print(f"  UI: {MLFLOW_TRACKING_URI}")

print(f"\nNext Steps:")
print(f"  1. Review results in MLflow UI")
print(f"  2. Deploy best model as REST API")
print(f"  3. Set up monitoring with Evidently")
print(f"  4. Create Airflow pipeline for automation")
print("=" * 80)